[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/pharmacoforge_smoke_test.ipynb)

# PharmacoForge smoke test

**Sandbox notebook - not workshop material.**

The only question this notebook answers is: *does PharmacoForge actually run in Colab today?*

PharmacoForge is a diffusion model that generates 3D pharmacophores for a protein pocket
([Flynn et al., Front. Bioinform. 2025](https://doi.org/10.3389/fbinf.2025.1628800), Koes lab,
[repository](https://github.com/eflynn8/pharmacophore-diffusion)).

This notebook follows the authors' own
[PharmacoForge Colab](https://colab.research.google.com/drive/1XZViTC6BiNN1tF0PIhZ48j3wHocUklHJ)
step by step, with two deliberate changes: their file-upload cells need a human, so we download
a small public test case instead, and we leave out their last cell, which sends the result to
Pharmit and is a known open bug
([issue #30](https://github.com/eflynn8/pharmacophore-diffusion/issues/30)).

## Before you run this

Set the runtime to **T4 GPU** *and* the **runtime version to 2025.07**
(*Runtime > Change runtime type*). Both matter, and the second one is not optional:

- PharmacoForge needs **DGL**, which is only published for **Python 3.8 to 3.12**.
- It also pins **PyTorch 2.4**, which has no Python 3.13 build either.
- Colab's current image ships **Python 3.13**, and `condacolab` makes conda packages
  importable from that *system* Python, so conda's Python has to match it. Installing an
  older Python into conda does not help.

So on today's default image the install cannot resolve at all. Pinning the image to 2025.07
is the authors' own instruction, and this notebook carries that pin in its metadata.

> **Note:** the repository has no `LICENSE` file, so it is all-rights-reserved. We clone it at
> run time and never copy any of it into this repository.

## What you will do

- Record what Python, PyTorch and GPU this Colab runtime has
- Install conda into Colab, then build the authors' environment with mamba
- Download the 16 MB pretrained checkpoint
- Prepare a small public receptor and reference ligand from the PDB
- Generate pharmacophores and check that real feature centres come out

## 1. Inspect the runtime

Record what we actually have before installing anything. `COLAB_RELEASE_TAG` is the Colab
image version, which is the thing the authors ask you to pin, so it is worth writing down
whatever the answer turns out to be.

In [ ]:
import os, shutil, subprocess, sys
print("Colab image:", os.environ.get("COLAB_RELEASE_TAG", "not on Colab"))
print("Python", sys.version.split()[0])
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch not installed")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())
else:
    print("No GPU. Set Runtime > Change runtime type > T4 GPU and run this cell again.")

## 2. Install conda

PharmacoForge cannot be installed with pip alone. Its `pyproject.toml` declares no
dependencies at all, and the graph library it uses, DGL, is only distributed through conda
channels. So the authors' notebook starts by putting conda into Colab with `condacolab`.

> **Note:** this cell **restarts the kernel**. Colab will say the session crashed. That is
> expected, not an error. Wait for it to come back, then carry on with the next cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Get PharmacoForge

Clone the repository. We pin it to a commit rather than following `main`, because the project
has no tagged releases and `main` has already broken this notebook's arguments once
([issue #25](https://github.com/eflynn8/pharmacophore-diffusion/issues/25)).

In [ ]:
COMMIT = "8ce6ab8130374093de88892d36f248ab794e56d7"  # main, 2026-01-06
REPO = "/content/pharmacophore-diffusion"
import os
if not os.path.exists(REPO):
    !git clone -q https://github.com/eflynn8/pharmacophore-diffusion {REPO}
    !git -C {REPO} checkout -q {COMMIT}
os.chdir(REPO)
!git -C {REPO} log -1 --oneline

## 4. Build the environment

These are the authors' own mamba lines, copied from their Colab. The pinning matters:
`pytorch==2.4` with `pytorch-cuda=12.1`, and DGL from the matching `th24_cu121` channel. Their
`install_things.sh` notes that *"as of last I did installs, pytorch 2.2.2 was not compatible
with dgl"*, so this pairing is not something to improvise on.

This takes several minutes and prints a lot. `pip install -e ./` at the end only registers the
package; it installs nothing, because the project declares no dependencies.

In [ ]:
!mamba install -q -y pytorch==2.4 torchdata=0.8.0 torchvision pytorch-cuda=12.1 -c pytorch -c nvidia
!mamba install -q -y -c dglteam/label/th24_cu121 dgl
!mamba install -q -y pytorch-cluster pytorch-scatter -c pyg
!mamba install -q -y -c conda-forge pytorch-lightning rdkit openbabel pandas biopython pydantic py3dmol
!pip install -q -e ./

Check that the two imports that usually break, `torch` and `dgl`, come up together and see
the GPU. If this cell fails, the rest cannot work, and the traceback is the useful thing to
report back.

In [ ]:
import torch, dgl
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("dgl", dgl.__version__)

## 5. Download the pretrained model

The weights are not in the repository. They live on the Koes lab server, and only the authors'
Colab records where. `--model_dir X` expects to find `X/checkpoints/last.ckpt` and
`X/config.yaml`, so the two files go to different places. The checkpoint is about 16 MB.

In [ ]:
model_dir = f"{REPO}/model_dir"
if not os.path.exists(model_dir):
    !wget -q -P {model_dir}/checkpoints https://bits.csb.pitt.edu/files/pharmacoforge/last.ckpt
    !wget -q -P {model_dir} https://bits.csb.pitt.edu/files/pharmacoforge/config.yaml
!ls -lh {model_dir} {model_dir}/checkpoints

## 6. Prepare a test case

PharmacoForge needs two inputs: a receptor `.pdb`, and a single-molecule `.sdf` whose
coordinates mark the pocket. The authors' notebook gets both from `files.upload()`. We use
trypsin with benzamidine bound (PDB [3PTB](https://www.rcsb.org/structure/3PTB)) instead: it is
small, public, and its pocket is one of the best characterised in structural biology.

Splitting a PDB into receptor and ligand is exactly the step the blue group will need for
CpABC1 and silymarin, so it is worth testing here.

> **Note:** PharmacoForge reads the SDF with `sanitize=False` and uses only the atom
> coordinates, so RDKit guessing the bonds is good enough *here*. A notebook that goes on to
> compare chemical features would need the bond orders rebuilt properly.

In [ ]:
import urllib.request
from rdkit import Chem

urllib.request.urlretrieve("https://files.rcsb.org/download/3PTB.pdb", "3ptb.pdb")
lines = open("3ptb.pdb").read().splitlines()

protein = [l for l in lines if l.startswith(("ATOM", "TER"))]
ligand = [l for l in lines if l.startswith("HETATM") and l[17:20] == "BEN"]
open("receptor.pdb", "w").write("\n".join(protein) + "\nEND\n")

mol = Chem.MolFromPDBBlock("\n".join(ligand), sanitize=False)
Chem.MolToMolFile(mol, "ligand.sdf")
print(f"{len(protein)} protein atoms | ligand: {mol.GetNumAtoms()} atoms, {Chem.MolToSmiles(mol)}")

## 7. Generate pharmacophores

This is the moment of truth. `--samples_per_pocket 3` with `--pharm_sizes 3 4 5` asks for three
pharmacophores of three, four and five feature centres. `--use_ref_lig_com` starts the
diffusion at the reference ligand's centre of mass.

> **Note:** the README's own example puts a literal `receptor_file` token before the path. That
> is a bug: `receptor_file` is the *positional* argument, so copying the README verbatim fails
> with an unrecognised-argument error. The form below is the one their Colab builds.

In [ ]:
!python generate_pharmacophores.py receptor.pdb \
  --ref_ligand_file ligand.sdf \
  --model_dir model_dir \
  --samples_per_pocket 3 --pharm_sizes 3 4 5 \
  --use_ref_lig_com --seed 42

## 8. Read the result

Output goes to `generated_pharms/<receptor name>/pharms.xyz`. It looks like an XYZ file but has
no comment line: just a count, then that many rows, repeated once per sample.

Feature types are smuggled in as fake chemical elements. The mapping comes from
`pharmacoforge/analysis/pharm_builder.py` and the order in the model's `config.yaml`.

In [ ]:
import pandas as pd

ELEMENT_TO_FEATURE = {"P": "Aromatic", "S": "HydrogenDonor", "F": "HydrogenAcceptor",
                      "N": "PositiveIon", "O": "NegativeIon", "C": "Hydrophobic"}

def parse_pharms(path):
    """Read a PharmacoForge .xyz into a table of one row per feature centre."""
    rows, lines, sample = [], open(path).read().split("\n"), 0
    while lines and lines[0].strip():
        n = int(lines[0])
        for line in lines[1:n + 1]:
            elem, x, y, z = line.split()
            rows.append({"sample": sample, "feature": ELEMENT_TO_FEATURE[elem],
                         "x": float(x), "y": float(y), "z": float(z)})
        lines, sample = lines[n + 1:], sample + 1
    return pd.DataFrame(rows)

pharms = parse_pharms("generated_pharms/receptor/pharms.xyz")
print(f"{pharms['sample'].nunique()} samples, {len(pharms)} feature centres")
pharms

## 9. Look at them in the pocket

The viewer code is the authors', lightly trimmed. Feature centres are wireframe spheres, with
the colours their notebook uses: aromatic purple, donor white, acceptor orange, hydrophobic
green, negative red, positive blue.

Seeing the centres sit *inside* the pocket, rather than drifting off into solvent, is the real
check that the model did something sensible.

In [ ]:
import py3Dmol

COLOURS = {"Aromatic": "purple", "HydrogenDonor": "0xf0f0f0", "HydrogenAcceptor": "orange",
           "Hydrophobic": "green", "NegativeIon": "red", "PositiveIon": "blue"}

v = py3Dmol.view(width="100%", height=450)
v.addModel(open("generated_pharms/receptor/pocket.pdb").read(), "pdb")
v.setStyle({"cartoon": {"color": "spectrum"}, "stick": {"radius": 0.1}})
for _, f in pharms.iterrows():
    v.addSphere({"center": {"x": f.x, "y": f.y, "z": f.z}, "radius": 0.9,
                 "color": COLOURS[f.feature], "wireframe": True})
v.zoomTo()
v.show()

## Summary

- If section 7 finished and section 8 printed a table of feature centres, PharmacoForge works
  in Colab on the current image, and the blue group can use it on CpABC1.
- If it failed, the likely culprit is section 4: the torch 2.4 / CUDA 12.1 / DGL pinning against
  whatever Colab ships today. Copy the traceback back into the chat. The first fallback is to
  set the runtime version to **2025.07** as the authors' notebook asks, and try again.
- Record here what actually happened, with the versions section 1 and section 4 printed.

Known rough edges, all of them upstream:

- The repository has no `LICENSE`, and no tagged release, so we pin a commit.
- The README's first usage example is wrong (a stray `receptor_file` token).
- The generated `.xyz` cannot be fed to Pharmit directly
  ([issue #27](https://github.com/eflynn8/pharmacophore-diffusion/issues/27)), and the authors'
  "load into Pharmit" cell is broken in Colab
  ([issue #30](https://github.com/eflynn8/pharmacophore-diffusion/issues/30)). For screening,
  their own advice is to use the web app at https://bits.csb.pitt.edu/forge/.

**Next:** if this passes, build `blue_pharmacophore_generation.ipynb` on the real
CpABC1-silymarin complex, where the ligand needs its bond orders rebuilt properly so its
features can be compared with the generated ones.